In [1]:
import numpy as np
from environments.simple_trading_env import SimpleTradingEnv
from stable_baselines3 import PPO
import pandas as pd
import torch

# === CONFIGURATION ===
DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_PATH = "trading_bot"

# Load data
df = pd.read_pickle(DATA_PATH)
print(f'✓ Loaded {len(df):,} rows for {DATA_SYMBOL} {DATA_TIMEFRAME}')

# Test on unseen data
total_timesteps = 1000
test_start = 121_072
test_data = df.iloc[test_start:test_start + total_timesteps].reset_index(drop=True)
print(f'✓ Testing on rows {test_start:,} to {test_start + total_timesteps:,}')

# Create test environment
test_env = SimpleTradingEnv(test_data)

# Load trained model
model = PPO.load(MODEL_PATH, env=test_env, device="cuda")
print(f'✓ Loaded model from {MODEL_PATH}\n')

# === FEATURE ACTIVATION TRACKING ===
extractor = model.policy.features_extractor
HOOKABLE_PATTERNS = ['_cnn', '_output', '_encoder', '_transformer', '_mlp', '_vp']

# Auto-discover all hookable modules
available_features = {}
for attr_name in dir(extractor):
    if attr_name.startswith('_'):
        continue
    if any(pattern in attr_name for pattern in HOOKABLE_PATTERNS):
        attr = getattr(extractor, attr_name)
        if isinstance(attr, torch.nn.Module):
            display_name = attr_name.replace('_', ' ').title().replace(' ', '_')
            available_features[attr_name] = display_name

✓ Loaded 264,323 rows for BTCUSDT 5m
✓ Testing on rows 121,072 to 122,072
Info: Dropped 99 rows due to NaNs after adding indicators.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
✓ Loaded model from trading_bot



In [2]:
from copy import deepcopy

# Setup hooks to capture activations
activations = {name: [] for name in available_features.keys()}

def make_hook(feature_name):
    def hook(module, input, output):
        # Capture mean absolute activation
        if isinstance(output, torch.Tensor):
            activations[feature_name].append(output.abs().mean().item())
    return hook

# Register hooks
hooks = []
for attr_name in available_features.keys():
    module = getattr(extractor, attr_name)
    hook = module.register_forward_hook(make_hook(attr_name))
    hooks.append(hook)

# Run evaluation
obs, _ = test_env.reset()
done = False
truncated = False
total_reward = 0
episode_rewards = []
step_count = 0
last_env = None

while not done and not truncated:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = test_env.step(action)
    total_reward += reward
    episode_rewards.append(reward)
    step_count += 1

    # Copy last environment state before fishing out or else will be resetted
    if test_env.data_len == info.get('step') + 1:
        last_env = deepcopy(test_env)

# Remove hooks
for hook in hooks:
    hook.remove()

# Print results
print("\n" + "="*80)
print("EVALUATION REPORT")
print("="*80)

print("\n📊 REWARD STATISTICS:")
print(f"   Total Reward:       {total_reward:+8.2f}")
print(f"   Min Reward:         {min(episode_rewards):+8.2f}")
print(f"   Max Reward:         {max(episode_rewards):+8.2f}")
print(f"   Avg Reward/Step:  {total_reward/step_count:+8.4f}")
print(f"   Total Steps:         {step_count:>6}")

# Feature activations
feature_stats = {name: np.mean(acts) if acts else 0.0 for name, acts in activations.items()}
sorted_features = sorted(feature_stats.items(), key=lambda x: x[1], reverse=True)

print(f"\n  Top 10 Most Active Features:")
for i, (name, avg_activation) in enumerate(sorted_features[:10], 1):
    display_name = available_features[name]
    print(f"    {i:2d}. {display_name:30s} : {avg_activation:.4f}")


EVALUATION REPORT

📊 REWARD STATISTICS:
   Total Reward:          -3.70
   Min Reward:            -0.50
   Max Reward:            +0.05
   Avg Reward/Step:   -0.0060
   Total Steps:            612

  Top 10 Most Active Features:
     1. Vp_Bins_Cnn                    : 0.5847
     2. Micro_Spatial_Cnn_Global       : 0.3537
     3. Position_Encoder               : 0.2299
     4. Vp_Levels_Binary_Mlp           : 0.1956
     5. Vp_Levels_Continuous_Mlp       : 0.1900
     6. Account_Encoder                : 0.1510
     7. Micro_Spatial_Cnn_Medium       : 0.1252
     8. Meso_Cnn                       : 0.0993
     9. Macro_Cnn                      : 0.0912
    10. Micro_Temporal_Cnn             : 0.0566


In [3]:
# === 4. COMPLETED TRADES ===
completed_trades = last_env.broker.trade_history
closed_trades = [t for t in completed_trades if t.get('status') == 'CLOSED']

if len(closed_trades) > 0:
    total_pnl = sum(t.get('pnl', 0) for t in closed_trades)
    total_commission = sum(t.get('commission', 0) for t in closed_trades)
    wins = [t for t in closed_trades if t.get('pnl', 0) > 0]
    losses = [t for t in closed_trades if t.get('pnl', 0) <= 0]
    
    print(f"\n📋 TRADE SUMMARY:")
    print(f"   Total Trades:  {len(closed_trades):>5}")
    print(f"   Wins:          {len(wins):>5} ({len(wins)/len(closed_trades)*100:5.1f}%)")
    print(f"   Losses:        {len(losses):>5} ({len(losses)/len(closed_trades)*100:5.1f}%)")
    print(f"   Total PnL:     ${total_pnl:>+10,.2f}")
    print(f"   Total Comm:    ${total_commission:>10,.2f}")
    print(f"   Net PnL:       ${total_pnl - total_commission:>+10,.2f}")
    print(f"   Avg PnL:       ${total_pnl/len(closed_trades):>+10,.2f}")
    if wins:
        print(f"   Avg Win:       ${sum(t['pnl'] for t in wins)/len(wins):>+10,.2f}")
    if losses:
        print(f"   Avg Loss:      ${sum(t['pnl'] for t in losses)/len(losses):>+10,.2f}")
    
    # Exit reason breakdown
    exit_reasons = {}
    for t in closed_trades:
        reason = t.get('reason', 'Unknown')
        exit_reasons[reason] = exit_reasons.get(reason, 0) + 1
    
    print(f"\n📊 EXIT REASONS:")
    for reason, count in sorted(exit_reasons.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(closed_trades)) * 100
        print(f"   {reason:20s}: {count:3d} ({pct:5.1f}%)")
    
    # === 5. DETAILED TRADE TABLE ===
    print("\n" + "="*80)
    print("DETAILED TRADE HISTORY")
    print("="*80)
    
    # Create trades DataFrame (all trades, including last open if any)
    all_trades = last_env.broker.trade_history
    trade_rows = []
    for i, t in enumerate(all_trades, 1):
        trade_rows.append({
            '#': i,
            'Status': t.get('status', 'OPEN'),
            'Step Open': t.get('step_open', 0),
            'Step Close': t.get('step_close', None),
            'Duration': t.get('duration', None),
            'Dir': 'LONG' if t.get('direction', 1) == 1 else 'SHORT',
            'Entry': t.get('entry_price', 0),
            'Exit': t.get('exit_price', None),
            'PnL': t.get('pnl', None),
            'PnL%': t.get('pnl_percent', None),
            'Comm': t.get('commission', 0),
            'Reason': t.get('reason', 'N/A'),
        })
    
    import pandas as pd
    trades_df = pd.DataFrame(trade_rows)
    
    def color_pnl(val):
        if val is None:
            return ''
        if val > 0:
            return 'background-color: #90EE90; color: black'
        elif val < 0:
            return 'background-color: #FF6B6B; color: black'
        else:
            return 'color: black'
    
    styled_df = trades_df.style.format({
        'Entry': '${:,.2f}',
        'Exit': '${:,.2f}',
        'PnL': '${:+,.2f}',
        'PnL%': '{:+.2f}%',
        'Comm': '${:.2f}'
    }).map(color_pnl, subset=['PnL'])
    
    display(styled_df)

else:
    print(f"\n⚠️  NO COMPLETED TRADES")



📋 TRADE SUMMARY:
   Total Trades:      8
   Wins:              0 (  0.0%)
   Losses:            8 (100.0%)
   Total PnL:     $   -839.58
   Total Comm:    $    286.90
   Net PnL:       $ -1,126.48
   Avg PnL:       $   -104.95
   Avg Loss:      $   -104.95

📊 EXIT REASONS:
   SL                  :   8 (100.0%)

DETAILED TRADE HISTORY


,#,Status,Step Open,Step Close,Duration,Dir,Entry,Exit,PnL,PnL%,Comm,Reason
0,1,CLOSED,509,520,11,LONG,"$66,210.03","$66,106.58",$-112.70,-1.13%,$47.95,SL
1,2,CLOSED,539,545,6,LONG,"$65,174.01","$64,955.92",$-104.22,-1.06%,$22.03,SL
2,3,CLOSED,547,571,24,LONG,"$64,832.00","$64,577.26",$-102.24,-1.05%,$18.55,SL
3,4,CLOSED,572,711,139,LONG,"$64,610.13","$64,371.97",$-101.17,-1.05%,$19.51,SL
4,5,CLOSED,712,712,0,LONG,"$64,506.15","$64,397.00",$-106.17,-1.12%,$42.07,SL
5,6,CLOSED,713,758,45,LONG,"$64,436.00","$64,323.94",$-104.31,-1.12%,$40.33,SL
6,7,CLOSED,759,759,0,LONG,"$64,326.01","$64,236.69",$-105.53,-1.15%,$49.82,SL
7,8,CLOSED,760,769,9,LONG,"$64,201.63","$64,107.92",$-103.23,-1.14%,$46.64,SL


## Trading Chart Visualization

Interactive candlestick chart with position markers and equity curve.

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def create_trading_chart(env):
    """
    Create candlestick chart with position markers using trades list.
    Uses env_data (not test_data) because env drops NaN rows during initialization.
    """
    env_history = env.history
    env_data = env.data
    trades = env.broker.trade_history

    # Create figure with 3 subplots
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        subplot_titles=('Price & Trading Activity', 'Account Equity', 'Rewards'),
        row_heights=[0.5, 0.25, 0.25]
    )

    # Add candlestick using env_data (which has correct indices)
    fig.add_trace(
        go.Candlestick(
            x=env_data.index,
            open=env_data['open'],
            high=env_data['high'],
            low=env_data['low'],
            close=env_data['close'],
            name='Price',
            increasing_line_color='#26a69a',
            decreasing_line_color='#ef5350'
        ),
        row=1, col=1
    )

    # Add trade entry/exit markers and SL/TP lines
    for trade in trades:
        if trade.get('status') != 'CLOSED':
            continue
        step_open = trade.get('step_open')
        step_close = trade.get('step_close')
        entry_price = trade.get('entry_price')
        exit_price = trade.get('exit_price')
        direction = 'LONG' if trade.get('direction', 1) == 1 else 'SHORT'
        sl = trade.get('sl_price')
        tp = trade.get('tp_price')
        pnl = trade.get('pnl', 0)

        # Entry marker
        color_entry = '#00ff00' if direction == 'LONG' else '#ff0000'
        symbol_entry = 'triangle-up' if direction == 'LONG' else 'triangle-down'
        fig.add_trace(
            go.Scatter(
                x=[step_open],
                y=[entry_price],
                mode='markers',
                marker=dict(size=15, color=color_entry, symbol=symbol_entry, line=dict(width=2, color='white')),
                name=f'{direction} Open',
                showlegend=False,
                hovertext=f'{direction} ENTRY<br>Step: {step_open}<br>Price: ${entry_price:,.2f}',
                hoverinfo='text'
            ),
            row=1, col=1
        )

        # Exit marker
        color_exit = '#90EE90' if pnl > 0 else '#FF6B6B'
        fig.add_trace(
            go.Scatter(
                x=[step_close],
                y=[exit_price],
                mode='markers',
                marker=dict(size=12, color=color_exit, symbol='x', line=dict(width=2, color='black')),
                name=f'{direction} Close',
                showlegend=False,
                hovertext=f'{direction} EXIT<br>Step: {step_close}<br>Price: ${exit_price:,.2f}<br>PnL: ${pnl:+,.2f}',
                hoverinfo='text'
            ),
            row=1, col=1
        )

        # SL/TP lines (draw only if open and close are not the same step)
        if sl is not None:
            fig.add_shape(
                type='line',
                x0=step_open, x1=step_close,
                y0=sl, y1=sl,
                line=dict(color='red', width=1, dash='dash'),
                row=1, col=1
            )
        if tp is not None:
            fig.add_shape(
                type='line',
                x0=step_open, x1=step_close,
                y0=tp, y1=tp,
                line=dict(color='green', width=1, dash='dash'),
                row=1, col=1
            )

    # Add equity curve
    steps = [s.get('step') for s in env_history]
    equity = [s.get('equity', 0) for s in env_history]

    fig.add_trace(
        go.Scatter(
            x=steps,
            y=equity,
            mode='lines',
            name='Equity',
            line=dict(color='#2196F3', width=2),
            fill='tozeroy',
            fillcolor='rgba(33, 150, 243, 0.1)'
        ),
        row=2, col=1
    )

    # Add rewards
    rewards = [s.get('reward', 0) for s in env_history]
    reward_colors = ['#90EE90' if r > 0 else '#FF6B6B' if r < 0 else '#888888' for r in rewards]

    fig.add_trace(
        go.Bar(
            x=steps,
            y=rewards,
            name='Reward',
            marker=dict(
                color=reward_colors,
                line=dict(width=0)
            ),
            hovertemplate='Step: %{x}<br>Reward: %{y:.4f}<extra></extra>'
        ),
        row=3, col=1
    )

    # Update layout
    fig.update_layout(
        title='Trading Activity Visualization',
        xaxis3_title='Step',
        yaxis_title='Price ($)',
        yaxis2_title='Equity ($)',
        yaxis3_title='Reward',
        height=1000,
        template='plotly_dark',
        hovermode='closest',
        showlegend=False
    )

    fig.update_xaxes(rangeslider_visible=False)

    return fig

# Generate chart using environment's data (not test_data)
print("\n" + "="*80)
print("Creating Trading Chart...")
print("="*80)
fig = create_trading_chart(last_env)
fig.show()
print("\n✓ Chart complete!")


Creating Trading Chart...



✓ Chart complete!
